Knowledge Distillation

In [29]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [30]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


Transforming the Dataset

In [31]:
transforms_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

transforms_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])


train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_test)

Train/Test Split

In [32]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

Model Definitions

In [33]:
# Teacher
class DeepNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Student
class LightNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

Training

In [34]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

def test(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

Cross Entropy Runs

Teacher Model Training

In [35]:
torch.manual_seed(42)
nn_deep = DeepNN(num_classes=10).to(device)
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_deep = test(nn_deep, test_loader, device)

Epoch 1/10, Loss: 1.5637473060042046
Epoch 2/10, Loss: 1.1645725299330318
Epoch 3/10, Loss: 0.9653376305804533
Epoch 4/10, Loss: 0.8507408453985248
Epoch 5/10, Loss: 0.7776272745083666
Epoch 6/10, Loss: 0.732624377893365
Epoch 7/10, Loss: 0.6885693484864881
Epoch 8/10, Loss: 0.6522845820240353
Epoch 9/10, Loss: 0.6273067782602042
Epoch 10/10, Loss: 0.6053646427133809
Test Accuracy: 79.61%


In [36]:
torch.manual_seed(42)
nn_light = LightNN(num_classes=10).to(device)
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light = test(nn_light, test_loader, device)

Epoch 1/10, Loss: 1.636555151256454
Epoch 2/10, Loss: 1.378120242177373
Epoch 3/10, Loss: 1.2741626013270424
Epoch 4/10, Loss: 1.1973345329999314
Epoch 5/10, Loss: 1.1398659822581065
Epoch 6/10, Loss: 1.0924492240561854
Epoch 7/10, Loss: 1.0532908316158578
Epoch 8/10, Loss: 1.0191575497617502
Epoch 9/10, Loss: 0.9983406630928254
Epoch 10/10, Loss: 0.9732208834279834
Test Accuracy: 68.32%


In [37]:
torch.manual_seed(42)
nn_light2 = LightNN(num_classes=10).to(device)
train(nn_light2, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light2 = test(nn_light2, test_loader, device)

Epoch 1/10, Loss: 1.6359483204839174
Epoch 2/10, Loss: 1.375329258191921
Epoch 3/10, Loss: 1.2725329136909427
Epoch 4/10, Loss: 1.1969369463908397
Epoch 5/10, Loss: 1.1363275776738706
Epoch 6/10, Loss: 1.091999145115123
Epoch 7/10, Loss: 1.053287763272405
Epoch 8/10, Loss: 1.0213522682409457
Epoch 9/10, Loss: 0.9961025844449583
Epoch 10/10, Loss: 0.9751765636531898
Test Accuracy: 68.48%


In [38]:
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy: {test_accuracy_light:.2f}%")
print(f"Student accuracy: {test_accuracy_light2:.2f}%")

Teacher accuracy: 79.61%
Student accuracy: 68.32%
Student accuracy: 68.48%


Knowledge Distillation

In [39]:
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    kl_loss = nn.KLDivLoss(reduction="batchmean")
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.eval()
    student.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)
            soft_targets_loss = kl_loss(soft_prob, soft_targets) * (T ** 2)

            label_loss = ce_loss(student_logits, labels)
            loss = (soft_target_loss_weight * soft_targets_loss) + (ce_loss_weight * label_loss)

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [40]:
train_knowledge_distillation(teacher=nn_deep, student=nn_light, train_loader=train_loader, epochs=20, learning_rate=0.001, T=3, soft_target_loss_weight=0.3, ce_loss_weight=0.7, device=device)
test_accuracy_light_ce_and_kd = test(nn_light, test_loader, device)

print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_light2:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd:.2f}%")

Epoch 1/20, Loss: 1.1197162105909089
Epoch 2/20, Loss: 1.0775008520201954
Epoch 3/20, Loss: 1.043231611056706
Epoch 4/20, Loss: 1.0259439303442035
Epoch 5/20, Loss: 1.0026814269897577
Epoch 6/20, Loss: 0.9864310512457357
Epoch 7/20, Loss: 0.9711553286713408
Epoch 8/20, Loss: 0.9488890881428633
Epoch 9/20, Loss: 0.940451392737191
Epoch 10/20, Loss: 0.9318330559279303
Epoch 11/20, Loss: 0.9217763640691558
Epoch 12/20, Loss: 0.908267001056915
Epoch 13/20, Loss: 0.9053978026675447
Epoch 14/20, Loss: 0.8916500087284371
Epoch 15/20, Loss: 0.8892671645754744
Epoch 16/20, Loss: 0.8759570074508257
Epoch 17/20, Loss: 0.8717535430817958
Epoch 18/20, Loss: 0.875493382249037
Epoch 19/20, Loss: 0.8599892176325669
Epoch 20/20, Loss: 0.8535772121470907
Test Accuracy: 74.98%
Teacher accuracy: 79.61%
Student accuracy without teacher: 68.48%
Student accuracy with CE + KD: 74.98%
